In [6]:
#Videodaki sırayı izleyerek Neuron, Layer ve MLP sınıflarını kur. Parametreleri tek listede topla, videodaki küçük veri setiyle eğit, loss'un adım adım düştüğünü göster. Her adımda gradient'leri sıfırlamayı unutma, videodaki meşhur bug. (1:43:55 - 2:14:03)
#radd added in order to calc sse
import math
class Value:
    def __init__(self,data,_children=(),_op="", label=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda:None #leaf node
        self._prev = set(_children)
        self._op=_op
        self.label= label

    def __repr__(self):
        return f"Value(data={self.data})" 

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other) #solution for Value + int
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += 1 * out.grad #accumulates the grad
            other.grad += 1 * out.grad
        out._backward = _backward
        
        return out
    
    def __radd__(self, other): #to solve int + value
        return self + other
        
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other) #Solution for Value * int
        out = Value(self.data * other.data, (self, other),"*")
            

        def _backward():
            self.grad += other.data * out.grad #accumulates the grad
            other.grad += self.data * out.grad
            
        out._backward = _backward
        return out

    def __pow__(self, other): 
        assert isinstance(other, (int,float))
        out = Value(self.data**other, (self,),f'**{other}')

        def _backward(): #x^n's derrivative
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward

        return out
    
    def __rmul__(self, other): #solution for int * value 
        return self * other

    def __truediv__(self,other): #self / other
        return self * other**-1

    def __neg__(self):
        return self * -1

    def __sub__(self,other):
        return self + (-other)
        
    def tanh(self):
        x=self.data
        t = (math.exp(2*x) - 1) / (math.exp (2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        
        return out

    def exp(self): #e^self.data
        x = self.data
        out = Value(math.exp(x), (self, ), 'exp')        

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        
        return out
        
    def backward(self):
        #sorting topo order
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad= 1.0  #first nodes grad = 1.0
        for node in reversed(topo):
            node._backward()

In [11]:
#new classes
import random
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
    
    def __call__(self,x):

        act = sum((wi*xi for wi, xi in zip(self.w,x)), self.b)
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]
        
class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range (nout)]

    def __call__(self,x):
        outs = [n(x) for n in self.neurons]
        return outs [0] if len (outs) == 1 else outs #if output layer has only 1 neuron just give it.

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range (len(nouts))]

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x
        
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
        
n = MLP(3, [4,4,1])

In [12]:
#small dataset
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]
ypred = [n(x) for x in xs]
ypred
   

[Value(data=-0.86828676577632),
 Value(data=-0.9509412203668378),
 Value(data=-0.8931052058154216),
 Value(data=-0.9805953505652998)]

In [13]:
#decreasing loss part with our small dataset.
for k in range(20):
    #generating preds & losses
    ypred=[n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip (ys,ypred))

    #backward pass 
    for p in n.parameters():
        p.grad= 0.0
    loss.backward()

    for p in n.parameters():
        p.data += -0.05 * p.grad

    print(k, loss.data)


0 7.427086642738683
1 6.251328510623828
2 3.8587664022058723
3 2.159347374782373
4 1.2197519059205584
5 0.7126285032881994
6 0.47157983252694263
7 0.33614421550834295
8 0.2606817123302414
9 0.2119503844214681
10 0.17751797462379693
11 0.15201350079419218
12 0.13243843540713246
13 0.11698703867071669
14 0.1045109819328029
15 0.09424718355386344
16 0.08566986988899289
17 0.07840562578550872
18 0.07218232735341504
19 0.06679722738882041


In [14]:
ypred

[Value(data=0.9140538067922487),
 Value(data=-0.9135770243086959),
 Value(data=-0.8551306748635378),
 Value(data=0.8240612971257246)]